In [2]:
pip install sentence-transformers transformers torch httpx openai networkx pandas protobuf==5.29.5

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install "mcp[cli]"


  Attempting uninstall: click

    Found existing installation: click 8.1.8

    Uninstalling click-8.1.8:

      Successfully uninstalled click-8.1.8

   ---------------------------------------- 0/2 [click]
  Attempting uninstall: typer
   ---------------------------------------- 0/2 [click]
    Found existing installation: typer 0.15.4
   ---------------------------------------- 0/2 [click]
   -------------------- ------------------- 1/2 [typer]
    Uninstalling typer-0.15.4:
   -------------------- ------------------- 1/2 [typer]
      Successfully uninstalled typer-0.15.4
   -------------------- ------------------- 1/2 [typer]
   -------------------- ------------------- 1/2 [typer]
   ---------------------------------------- 2/2 [typer]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
together 1.5.26 requires typer<0.16,>=0.9, but you have typer 0.24.1 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# !pip install -q requests beautifulsoup4 trafilatura pandas httpx openai

import os
import re
import json
import time
import httpx
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
import trafilatura
from bs4 import BeautifulSoup, Tag
from openai import OpenAI

# =========================================================
# 0) LLM CONFIG (TokenFactory)
# =========================================================

# TokenFactory API credentials (same as ExecutionAgent)
LLM_API_KEY = os.getenv("LLM_API_KEY", "sk-54aa2aca90034799817c91c16e83f3e2")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://tokenfactory.esprit.tn/api")
LLM_SUMMARIZER_MODEL = os.getenv("LLM_SUMMARIZER_MODEL", "hosted_vllm/Llama-3.1-70B-Instruct")
VERIFY_SSL = os.getenv("LLM_VERIFY_SSL", "false").lower() in ["true", "1", "yes"]

SUMMARIZER_MAX_TOKENS = int(os.getenv("SUMMARIZER_MAX_TOKENS", "400"))
MODEL_CALL_SLEEP_SECONDS = 0.5

# =========================================================
# 1) LLM CLIENT (TokenFactory)
# =========================================================

class LLMClient:
    """
    OpenAI-compatible TokenFactory client for summarization.
    """

    def __init__(self, api_key: str, base_url: str):
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")
        self._healthchecked = False
        self._health_ok = False

        http_client = httpx.Client(
            verify=VERIFY_SSL,
            timeout=120.0
        )

        self.client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            http_client=http_client,
        )

    def healthcheck(self, force: bool = False) -> bool:
        if not self.api_key or self.api_key == "YOUR_TOKEN_FACTORY_KEY":
            return False
        if self._healthchecked and not force:
            return self._health_ok

        try:
            _ = self.client.chat.completions.create(
                model=LLM_SUMMARIZER_MODEL,
                messages=[{"role": "user", "content": "hi"}],
                max_tokens=5,
                temperature=0.1,
            )
            self._health_ok = True
            self._healthchecked = True
            print("[LLM healthcheck] ✓ API accessible")
            return True
        except Exception as e:
            self._health_ok = False
            self._healthchecked = True
            print(f"[LLM healthcheck] ✗ Failed: {type(e).__name__}: {str(e)[:150]}")
            return False

    def summarize_text(
        self,
        text: str,
        heading: str = "",
        max_tokens: int = 400,
        temperature: float = 0.3,
        max_retries: int = 2,
    ) -> str:
        """
        Summarize a section of text using TokenFactory LLM.
        Returns a concise summary (3-5 sentences).
        """
        
        system_prompt = """You are a concise summarization assistant. 
Your task is to summarize the given section content into 3-5 clear, actionable sentences.
Focus on the key points and practical implications.
Avoid jargon when possible, and be direct."""

        user_prompt = f"""Section heading: {heading}

Content to summarize:
{text}

Provide a clear, concise summary in 3-5 sentences."""

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=LLM_SUMMARIZER_MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens,
                )

                summary = response.choices[0].message.content.strip()
                time.sleep(MODEL_CALL_SLEEP_SECONDS)
                return summary

            except Exception as e:
                if attempt < max_retries - 1:
                    wait_s = min(10, (2 ** attempt) * 2)
                    print(f"[Summarization] Retry in {wait_s}s: {type(e).__name__}")
                    time.sleep(wait_s)
                else:
                    print(f"[Summarization] Failed after {max_retries} attempts: {type(e).__name__}: {str(e)[:100]}")
                    return ""

        return ""

# =========================================================
# 2) CONFIG
# =========================================================

SOURCES = [
    {
        "doc_id": "KB1",
        "title": "Minimum viable product (MVP): What is it & how to define one",
        "category": "product_execution",
        "url": "https://www.atlassian.com/agile/product-management/minimum-viable-product"
    },
    {
        "doc_id": "KB2",
        "title": "Create a work breakdown structure | Microsoft Learn",
        "category": "task_decomposition",
        "url": "https://learn.microsoft.com/en-us/dynamics365/project-operations/project-management/create-wbs"
    },
    {
        "doc_id": "KB2B",
        "title": "Work breakdown structures overview | Microsoft Learn",
        "category": "task_decomposition",
        "url": "https://learn.microsoft.com/en-us/dynamics365/project-operations/prod-pma/work-breakdown-structures"
    },
]

OUTPUT_DIR = "structured_kb_sections"
HTML_DIR = os.path.join(OUTPUT_DIR, "html")
JSON_DIR = os.path.join(OUTPUT_DIR, "json_per_url")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

HEADING_TAGS = ["h1", "h2", "h3", "h4"]
CONTENT_TAGS = ["p", "ul", "ol", "table", "pre", "blockquote"]

# Initialize LLM client (will be checked later)
llm_client = None
llm_available = False

# =========================================================
# 3) UTILS
# =========================================================

def safe_attr_str(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return " ".join(str(v) for v in value)
    return str(value)

def clean_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ").replace("\u200b", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_block_text(text: Optional[str]) -> str:
    text = clean_text(text)
    if not text:
        return ""

    noise_patterns = [
        r"skip to main content",
        r"table of contents",
        r"additional resources",
        r"was this page helpful\??",
        r"need help with this topic\??",
        r"ask learn",
        r"suggest a fix\??",
        r"this browser is no longer supported",
        r"upgrade to microsoft edge.*?technical support",
        r"feedback",
        r"previous version",
        r"print",
        r"theme",
        r"light\s+dark\s+high contrast",
    ]

    for pat in noise_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def safe_filename(name: str) -> str:
    return re.sub(r"[^\w\-\.]+", "_", str(name))[:180]

def is_microsoft_learn_url(url: str) -> bool:
    return "learn.microsoft.com" in url.lower()

def fetch_html(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=40)
    r.raise_for_status()
    return r.text

def extract_title_and_meta(soup):
    title = ""
    meta_description = ""

    try:
        if soup.title and soup.title.string:
            title = clean_text(soup.title.string)
    except Exception:
        pass

    try:
        meta = soup.find("meta", attrs={"name": "description"})
        if meta:
            meta_description = clean_text(meta.get("content"))
    except Exception:
        pass

    return title, meta_description

# =========================================================
# 4) MAIN CONTAINER EXTRACTION
# =========================================================

def remove_noise_from_container(container):
    if container is None:
        return container

    for tag in container.find_all([
        "script", "style", "noscript", "svg", "form", "iframe"
    ]):
        try:
            tag.decompose()
        except Exception:
            pass

    noise_keywords = [
        "footer", "nav", "menu", "cookie", "consent", "banner",
        "breadcrumb", "sidebar", "newsletter", "social", "share",
        "related", "recommend", "promo", "advert", "ads",
        "popup", "modal", "feedback", "helpful", "toolbar"
    ]

    for tag in container.find_all(True):
        try:
            attrs = " ".join([
                safe_attr_str(tag.get("class")),
                safe_attr_str(tag.get("id")),
                safe_attr_str(tag.get("role")),
                safe_attr_str(tag.get("aria-label")),
                safe_attr_str(tag.get("data-bi-name")),
            ]).lower()

            if any(k in attrs for k in noise_keywords):
                tag.decompose()
        except Exception:
            pass

    return container

def extract_main_container(soup, url=""):
    if soup is None:
        return None, "none"

    if is_microsoft_learn_url(url):
        selectors = [
            ("article", lambda s: s.find("article")),
            ("data-bi-name=content-page", lambda s: s.find(attrs={"data-bi-name": "content-page"})),
            ("role=main", lambda s: s.find(attrs={"role": "main"})),
            ("main", lambda s: s.find("main")),
            ("id=main", lambda s: s.find(id="main")),
            ("id=main-column", lambda s: s.find(id="main-column")),
        ]
        for label, fn in selectors:
            try:
                node = fn(soup)
                if node:
                    return node, f"microsoft_specific:{label}"
            except Exception:
                pass

    generic_candidates = [
        ("main", lambda s: s.find("main")),
        ("article", lambda s: s.find("article")),
        ("role_main", lambda s: s.find(attrs={"role": "main"})),
        ("id_main", lambda s: s.find(id="main")),
        ("body", lambda s: s.find("body")),
    ]

    for label, fn in generic_candidates:
        try:
            node = fn(soup)
            if node:
                return node, label
        except Exception:
            pass

    return soup, "full_soup"

# =========================================================
# 5) BLOCK CONVERSION
# =========================================================

def html_table_to_rows(tag: Tag) -> List[List[str]]:
    rows = []
    for tr in tag.find_all("tr"):
        cells = []
        for cell in tr.find_all(["th", "td"]):
            cell_txt = clean_block_text(cell.get_text(" ", strip=True))
            if cell_txt:
                cells.append(cell_txt)
        if cells:
            rows.append(cells)
    return rows

def tag_to_block(tag: Tag):
    if not isinstance(tag, Tag):
        return None

    name = tag.name.lower()

    if name == "p":
        txt = clean_block_text(tag.get_text(" ", strip=True))
        if len(txt) >= 25:
            return {"type": "paragraph", "text": txt}
        return None

    if name in ["ul", "ol"]:
        items = []
        for li in tag.find_all("li", recursive=False):
            li_txt = clean_block_text(li.get_text(" ", strip=True))
            if len(li_txt) >= 3:
                items.append(li_txt)
        if items:
            return {"type": "list", "items": items}
        return None

    if name == "table":
        rows = html_table_to_rows(tag)
        if rows:
            return {"type": "table", "rows": rows}
        return None

    if name == "pre":
        txt = clean_block_text(tag.get_text("\n", strip=True))
        if txt:
            return {"type": "preformatted", "text": txt}
        return None

    if name == "blockquote":
        txt = clean_block_text(tag.get_text(" ", strip=True))
        if txt:
            return {"type": "quote", "text": txt}
        return None

    return None

# =========================================================
# 6) SECTION BUILDING LOGIC
# =========================================================

def block_to_text(block: Dict[str, Any]) -> str:
    btype = block.get("type")

    if btype in ["paragraph", "preformatted", "quote"]:
        return clean_block_text(block.get("text", ""))

    if btype == "list":
        return " ; ".join(clean_block_text(x) for x in block.get("items", []) if clean_block_text(x))

    if btype == "table":
        rows = block.get("rows", [])
        return " ; ".join(" | ".join(clean_block_text(c) for c in row) for row in rows[:6])

    return ""

def classify_block(block: Dict[str, Any]) -> str:
    """
    Label sémantique simple pour former des mini-sections claires.
    """
    if not block:
        return "other"

    btype = block.get("type")

    if btype == "table":
        return "Table"

    if btype == "list":
        list_text = block_to_text(block).lower()

        if re.search(r"\b(step|steps|first|second|third|next|then|finally)\b", list_text):
            return "Steps"

        if re.search(r"\b(checklist|check list|ensure|make sure|required|requirements)\b", list_text):
            return "Checklist"

        return "Key points"

    text = block_to_text(block).lower()

    if re.search(r"\b(is|means|refers to|defined as|definition)\b", text):
        return "Definition"

    if re.search(r"\b(step|steps|first|second|third|next|then|finally|process|procedure|how to)\b", text):
        return "Steps"

    if re.search(r"\b(example|examples|for example|for instance|such as)\b", text):
        return "Examples"

    if re.search(r"\b(note|important|warning|remember|caution)\b", text):
        return "Notes"

    return "Explanation"

def make_heading_like_title(text: str, default_prefix: str = "Topic") -> str:
    text = clean_block_text(text)
    if not text:
        return default_prefix

    text = text.strip(":-• ")
    if len(text) > 90:
        text = text[:90].rsplit(" ", 1)[0]

    return text

def group_blocks_into_mini_sections(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Convert heading content into smaller clear semantic groups.
    """
    mini_sections = []
    current_blocks = []
    current_label = None

    for block in blocks:
        label = classify_block(block)

        if current_blocks and label != current_label:
            mini_sections.append({
                "title": current_label,
                "blocks": current_blocks
            })
            current_blocks = []

        current_blocks.append(block)
        current_label = label

    if current_blocks:
        mini_sections.append({
            "title": current_label,
            "blocks": current_blocks
        })

    return mini_sections

def iter_meaningful_nodes(container: Tag):
    for elem in container.find_all(HEADING_TAGS + CONTENT_TAGS):
        yield elem

def extract_document_sections(main_container):
    """
    Build:
    intro + sections based on headings + mini_sections inside each section
    """
    intro_blocks = []
    raw_sections = []
    current_section = None
    seen_fingerprints = set()

    for elem in iter_meaningful_nodes(main_container):
        if not isinstance(elem, Tag):
            continue

        if elem.name in HEADING_TAGS:
            heading_text = clean_block_text(elem.get_text(" ", strip=True))
            if len(heading_text) < 2:
                continue

            current_section = {
                "heading": heading_text,
                "level": elem.name,
                "content": []
            }
            raw_sections.append(current_section)
            continue

        block = tag_to_block(elem)
        if block is None:
            continue

        fp = json.dumps(block, ensure_ascii=False, sort_keys=True)
        if fp in seen_fingerprints:
            continue
        seen_fingerprints.add(fp)

        if current_section is None:
            intro_blocks.append(block)
        else:
            current_section["content"].append(block)

    final_sections = []
    for sec in raw_sections:
        if not sec["content"]:
            continue

        mini_sections = group_blocks_into_mini_sections(sec["content"])

        final_sections.append({
            "heading": sec["heading"],
            "level": sec["level"],
            "mini_sections": mini_sections
        })

    return intro_blocks, final_sections

# =========================================================
# 7) FALLBACK WITH CLEAR SECTIONS
# =========================================================

def build_fallback_sections_from_text(text):
    """
    Fallback si la structure HTML est faible :
    on découpe le texte en topics + mini_sections.
    """
    if not text:
        return [], []

    raw_parts = re.split(r"\n\s*\n+", text)
    raw_parts = [clean_block_text(p) for p in raw_parts if clean_block_text(p)]
    raw_parts = [p for p in raw_parts if len(p) >= 25]

    if not raw_parts:
        return [], []

    intro_blocks = [{"type": "paragraph", "text": raw_parts[0]}]
    sections = []

    current_paragraphs = []
    current_heading = None
    section_index = 1

    for part in raw_parts[1:]:
        # petite heuristique : texte court = possible heading
        if len(part) < 85 and part.count(".") <= 1:
            if current_paragraphs:
                blocks = [{"type": "paragraph", "text": p} for p in current_paragraphs]
                sections.append({
                    "heading": current_heading or f"Topic {section_index}",
                    "level": "fallback",
                    "mini_sections": group_blocks_into_mini_sections(blocks)
                })
                section_index += 1
                current_paragraphs = []

            current_heading = make_heading_like_title(part, default_prefix=f"Topic {section_index}")
        else:
            current_paragraphs.append(part)

    if current_paragraphs:
        blocks = [{"type": "paragraph", "text": p} for p in current_paragraphs]
        sections.append({
            "heading": current_heading or f"Topic {section_index}",
            "level": "fallback",
            "mini_sections": group_blocks_into_mini_sections(blocks)
        })

    return intro_blocks, sections

# =========================================================
# 8) SUMMARIZE SECTIONS (NEW)
# =========================================================

def summarize_section_content(section: Dict[str, Any]) -> str:
    """
    Summarize a section using LLM if available.
    Returns summary, or empty string if LLM unavailable.
    """
    global llm_client, llm_available

    if not llm_available or not llm_client:
        return ""

    # Gather all text from blocks in the section
    all_texts = []
    for mini_section in section.get("mini_sections", []):
        for block in mini_section.get("blocks", []):
            txt = block_to_text(block)
            if txt:
                all_texts.append(txt)

    if not all_texts:
        return ""

    section_content = " ".join(all_texts)
    if len(section_content) < 50:
        return ""

    heading = section.get("heading", "")
    
    # Call LLM to summarize
    summary = llm_client.summarize_text(
        text=section_content,
        heading=heading,
        max_tokens=SUMMARIZER_MAX_TOKENS,
    )
    
    return summary

# =========================================================
# 9) PAGE TO JSON
# =========================================================

def page_to_structured_json(source):
    url = source["url"]

    result = {
        "doc_id": source["doc_id"],
        "input_title": source["title"],
        "category": source["category"],
        "url": url,
        "site": "",
        "page_title": "",
        "meta_description": "",
        "main_selector": "",
        "intro": [],
        "sections": [],
        "stats": {},
        "error": None
    }

    try:
        html = fetch_html(url)

        html_filename = safe_filename(source["doc_id"] + "_" + source["title"]) + ".html"
        html_path = os.path.join(HTML_DIR, html_filename)
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html)

        soup = BeautifulSoup(html, "html.parser")
        page_title, meta_description = extract_title_and_meta(soup)
        main_container, main_selector = extract_main_container(soup, url=url)

        result["site"] = re.sub(r"^www\.", "", requests.utils.urlparse(url).netloc)
        result["page_title"] = page_title
        result["meta_description"] = meta_description
        result["main_selector"] = main_selector

        if main_container is not None:
            try:
                container_html = str(main_container)
                local_soup = BeautifulSoup(container_html, "html.parser")
            except Exception:
                local_soup = main_container

            local_soup = remove_noise_from_container(local_soup)
            intro_blocks, sections = extract_document_sections(local_soup)
        else:
            intro_blocks, sections = [], []

        total_blocks = len(intro_blocks) + sum(
            len(ms["blocks"])
            for sec in sections
            for ms in sec.get("mini_sections", [])
        )

        if total_blocks < 4:
            try:
                trafi_text = trafilatura.extract(
                    html,
                    include_links=False,
                    include_images=False,
                    include_tables=True,
                    no_fallback=False
                )
                trafi_text = trafi_text if trafi_text else ""
            except Exception:
                trafi_text = ""

            if trafi_text:
                fb_intro, fb_sections = build_fallback_sections_from_text(trafi_text)
                fb_total = len(fb_intro) + sum(
                    len(ms["blocks"])
                    for sec in fb_sections
                    for ms in sec.get("mini_sections", [])
                )
                if fb_total > total_blocks:
                    intro_blocks, sections = fb_intro, fb_sections

        result["intro"] = intro_blocks
        result["sections"] = sections

        # Add LLM summaries to each section
        if llm_available and llm_client:
            for sec in result.get("sections", []):
                summary = summarize_section_content(sec)
                if summary:
                    sec["summary"] = summary
                    print(f"[Summarized] {sec.get('heading', '')[:60]}")

        paragraph_count = 0
        list_count = 0
        table_count = 0
        pre_count = 0
        quote_count = 0
        mini_sections_count = 0

        for block in intro_blocks:
            btype = block.get("type")
            if btype == "paragraph":
                paragraph_count += 1
            elif btype == "list":
                list_count += 1
            elif btype == "table":
                table_count += 1
            elif btype == "preformatted":
                pre_count += 1
            elif btype == "quote":
                quote_count += 1

        for sec in sections:
            mini_sections_count += len(sec.get("mini_sections", []))
            for ms in sec.get("mini_sections", []):
                for block in ms.get("blocks", []):
                    btype = block.get("type")
                    if btype == "paragraph":
                        paragraph_count += 1
                    elif btype == "list":
                        list_count += 1
                    elif btype == "table":
                        table_count += 1
                    elif btype == "preformatted":
                        pre_count += 1
                    elif btype == "quote":
                        quote_count += 1

        result["stats"] = {
            "num_sections": len(sections),
            "num_mini_sections": mini_sections_count,
            "num_intro_blocks": len(intro_blocks),
            "num_paragraph_blocks": paragraph_count,
            "num_list_blocks": list_count,
            "num_table_blocks": table_count,
            "num_preformatted_blocks": pre_count,
            "num_quote_blocks": quote_count
        }

        return result

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"
        return result

# =========================================================
# 10) SAVE JSON
# =========================================================

def save_json(path: str, data: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# =========================================================
# 11) OPTIONAL FLATTENED RECORDS FOR KB
# =========================================================

def flatten_sections_to_records(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Transforme les sections / mini_sections en enregistrements simples pour KB.
    Includes LLM summaries if available.
    """
    records = []

    for sec_idx, sec in enumerate(doc.get("sections", []), start=1):
        section_heading = sec.get("heading", "")
        section_level = sec.get("level", "")
        section_summary = sec.get("summary", "")

        for mini_idx, mini in enumerate(sec.get("mini_sections", []), start=1):
            texts = []
            for block in mini.get("blocks", []):
                txt = block_to_text(block)
                if txt:
                    texts.append(txt)

            merged_text = "\n".join(texts).strip()
            if not merged_text:
                continue

            records.append({
                "doc_id": doc.get("doc_id"),
                "category": doc.get("category"),
                "url": doc.get("url"),
                "page_title": doc.get("page_title"),
                "section_id": f"{doc.get('doc_id')}_sec_{sec_idx}_mini_{mini_idx}",
                "section_heading": section_heading,
                "section_level": section_level,
                "section_summary": section_summary,
                "mini_section_title": mini.get("title", ""),
                "text": merged_text
            })

    return records

# =========================================================
# 12) RUN
# =========================================================

# Initialize LLM client
print("Initializing LLM client...")
llm_client = LLMClient(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
llm_available = llm_client.healthcheck()

if llm_available:
    print("✓ LLM summarization enabled")
else:
    print("⚠ LLM unavailable - sections will not be summarized")

all_docs = []
all_records = []

for src in SOURCES:
    print("=" * 100)
    print(f"Processing: {src['doc_id']} | {src['url']}")

    doc = page_to_structured_json(src)

    json_path = os.path.join(
        JSON_DIR,
        safe_filename(doc["doc_id"] + "_" + doc["input_title"]) + ".json"
    )
    save_json(json_path, doc)

    records = flatten_sections_to_records(doc)
    all_docs.append(doc)
    all_records.extend(records)

    print("Saved:", json_path)
    print("Title:", doc.get("page_title"))
    print("Main selector:", doc.get("main_selector"))
    print("Sections:", doc.get("stats", {}).get("num_sections", 0))
    print("Mini-sections:", doc.get("stats", {}).get("num_mini_sections", 0))
    print("Error:", doc.get("error"))

save_json(os.path.join(OUTPUT_DIR, "all_structured_documents.json"), all_docs)
save_json(os.path.join(OUTPUT_DIR, "all_kb_records.json"), all_records)

df = pd.DataFrame(all_records)
display(df.head(20))

print("\nDONE")
print("Documents:", len(all_docs))
print("KB records:", len(all_records))


Processing: KB1 | https://www.atlassian.com/agile/product-management/minimum-viable-product
Saved: structured_kb_sections\json_per_url\KB1_Minimum_viable_product_MVP_What_is_it_how_to_define_one.json
Title: Minimum viable product (MVP): What is it & how to start | Atlassian
Main selector: main
Sections: 22
Mini-sections: 42
Error: None
Processing: KB2 | https://learn.microsoft.com/en-us/dynamics365/project-operations/project-management/create-wbs
Saved: structured_kb_sections\json_per_url\KB2_Create_a_work_breakdown_structure_Microsoft_Learn.json
Title: Create a work breakdown structure | Microsoft Learn
Main selector: microsoft_specific:role=main
Sections: 17
Mini-sections: 36
Error: None
Processing: KB2B | https://learn.microsoft.com/en-us/dynamics365/project-operations/prod-pma/work-breakdown-structures
Saved: structured_kb_sections\json_per_url\KB2B_Work_breakdown_structures_overview_Microsoft_Learn.json
Title: Work breakdown structures overview | Microsoft Learn
Main selector: mic

,doc_id,category,url,page_title,section_id,section_heading,section_level,mini_section_title,text
0,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_1,Minimum viable product (MVP): What is it & how...,h1,Explanation,Get the free product discovery template\nAlign...
1,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_2,Minimum viable product (MVP): What is it & how...,h1,Steps,A minimum viable product (MVP) is the simplest...
2,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_3,Minimum viable product (MVP): What is it & how...,h1,Definition,A minimum viable product (MVP) is the simplest...
3,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_4,Minimum viable product (MVP): What is it & how...,h1,Explanation,"MVPs reduce risk, accelerate learning, and inf..."
4,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_5,Minimum viable product (MVP): What is it & how...,h1,Steps,Launch an MVP to quickly validate assumptions ...
5,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_1_mini_6,Minimum viable product (MVP): What is it & how...,h1,Definition,Did you know that Amazon started as a seller o...
6,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_2_mini_1,What is a minimum viable product (MVP)?,h2,Definition,Put in basic terms: the minimum viable product...
7,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_2_mini_2,What is a minimum viable product (MVP)?,h2,Explanation,âThe version of a new product which allows a...
8,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_3_mini_1,The importance and benefits of implementing MVPs,h3,Steps,The concept of an MVP comes from lean startup ...
9,KB1,product_execution,https://www.atlassian.com/agile/product-manage...,Minimum viable product (MVP): What is it & how...,KB1_sec_3_mini_2,The importance and benefits of implementing MVPs,h3,Definition,The benefits of focusing on an MVP and scaling...



DONE
Documents: 3
KB records: 168
